# 35 — Flattened Sequence Boosting
This notebook tests one hypothesis only: whether leakage-free L12 history supplied as flattened tabular features can reproduce the recorded Notebook 33 GRU signal. LightGBM family, dual known/masked experts, canonical parameters, seed and target remain fixed. No external data/package download is permitted.


## Frozen gates and leakage contract
Before results: A requires strict overall gain ≥0.002 with no worst-fold degradation; B masked gain ≥0.004 with known degradation ≤0.001; C blend gain ≥0.0015 with strict/space-time agreement; D reproduces ≥60% of recorded GRU gain and passes robust full validation. Every lag is location-wise `shift(k)` and must have source date strictly earlier than prediction date. Same-date values, validation targets, target priors, teacher forcing and future rows are forbidden. Full execution stops if estimated runtime exceeds 30 minutes or peak RAM exceeds 8 GB.


In [1]:
%run ../scripts/notebook35_runtime.py


Notebook35 start RAM GB 0.1832733154296875


# Notebook 35 final recommendation

1. Flattened lag pipeline leakage-free kuruldu mu? **Evet, pilot kapsamında.** Tüm laglar location içi `shift(k)` ile üretildi; `max(source_date) < prediction_date` doğrulandı. Target-derived feature yoktur.
2. En iyi window length hangisi? Pilot F3 ablation sonucu **L=6**; ana L12 karşılaştırması önceden sabitti.
3. Raw lag mı summary feature mı daha faydalı? L12 pilotunda **raw lag** (F1=0.640994, F2=0.647039).
4. Flattened LightGBM canonical modeli geçti mi? Aynı satırlardaki recorded canonical E0'a göre **True** (E0=0.772760, F1_L12=0.640994). Bu yalnız pilot kanıtıdır.
5. Masked RMSE ne kadar değişti? 0.808143 -> 0.659080; gain=+0.149063.
6. Robust validation kapıları geçildi mi? **Hayır.** Full strict/irregular/spatial/space-time validation çalıştırılmadı; tahmin 93.8 dakika ve 16.6 GB peak RAM, durum `not_run_performance_gate`.
7. GRU pilot kazancının ne kadarı yeniden üretildi? Gösterge oranı **172.8%**; farklı eğitim protokolleri nedeniyle d

## Results
Notebook 33 GRU entries are recorded pilot results, not recomputed observations. `full_validation_results.csv` explicitly distinguishes computed metrics from stopped work; missing full metrics are never inferred.


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
NB_ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'data/raw/Train.csv').exists()); NB_ART=NB_ROOT/'artifacts/notebook35'
for name in ['window_definition','lag_feature_report','cold_start_analysis','pilot_results','full_validation_results','diversity_analysis','gru_comparison','feature_importance','submission_report']:
 display(Markdown('### '+name)); display(pd.read_csv(NB_ART/f'{name}.csv').head(30))


### window_definition

,Window_length,Rows,Cache_SHA256,Build_seconds,Peak_RAM_GB,Source_date_assertion,Same_date_forbidden,Target_derived_features,Left_padding
0,3,43660,422ec15f19e8135f9b6d51bc2078c2a6a8540766b2716a...,59.164244,1.119759,True,True,False,True
1,6,43660,475a45c21d3f01312da422ad9cf3ffa602db6e439b1dc7...,59.112927,0.996006,True,True,False,True
2,12,43660,cfdc3c42cd6056ee6df9feb9b8161486677ae518f1af12...,48.712557,1.022743,True,True,False,True


### lag_feature_report

,Feature_set,Definition,Feature_count,Allowed,Reason
0,F1,raw lags 1..12 plus TWS missing masks,-1,True,NaN
1,F2,nine summaries per dynamic feature plus TWS mo...,see model feature counts,True,NaN
2,F3,F1 + F2,see model feature counts,True,NaN
3,F4,OOF-stable small subset,0,False,one pilot fold cannot establish fold stability


### cold_start_analysis

,Cold_start,Rows,Fraction,Flattened_RMSE,Masked_RMSE,Canonical_RMSE,Chosen_fallback
0,False,7576,0.999868,0.641025,0.659124,0.772806,flattened left-padding/native-missing; no cano...
1,True,1,0.000132,0.328233,0.328233,0.245777,flattened left-padding/native-missing; no cano...


... (8 outputs left out)

### feature_importance

,Feature_set,Expert,Feature,Importance_gain
0,E0_L12,known,TWS_t,144097.172748
1,F3_L3,known,TWS_t,142066.190278
2,F3_L6,known,TWS_t,136496.839783
3,F1_L12,known,TWS_t,131167.316645
4,F3_L12,known,TWS_t,123046.569263
5,F2_L12,known,TWS_t,120093.854074
6,F2_L12,masked,TWS_t_last_valid_value,108649.366883
7,F3_L6,masked,TWS_t_lag_1,103325.635751
8,F1_L12,masked,TWS_t_lag_1,95753.837770
9,F3_L12,masked,TWS_t_lag_1,87819.107512


### submission_report

,Submission,Created,Rows,Reason,Acceptance,Gate_A_pilot,Gate_B_pilot,Gate_C,Gate_D,SHA256
0,none,False,0,Robust full validation not completed; acceptan...,False,True,True,False,False,NaN


In [3]:
required=['window_definition.csv','lag_feature_report.csv','cold_start_analysis.csv','pilot_results.csv','full_validation_results.csv','diversity_analysis.csv','gru_comparison.csv','feature_importance.csv','submission_report.csv','final_recommendation.md']
assert not [x for x in required if not (NB_ART/x).exists()]
assert not list((NB_ROOT/'submissions').glob('submission_35*.csv'))
display(Markdown((NB_ART/'final_recommendation.md').read_text(encoding='utf-8')))
print('All required outputs verified; no submission created.')


# Notebook 35 final recommendation

1. Flattened lag pipeline leakage-free kuruldu mu? **Evet, pilot kapsamında.** Tüm laglar location içi `shift(k)` ile üretildi; `max(source_date) < prediction_date` doğrulandı. Target-derived feature yoktur.
2. En iyi window length hangisi? Pilot F3 ablation sonucu **L=6**; ana L12 karşılaştırması önceden sabitti.
3. Raw lag mı summary feature mı daha faydalı? L12 pilotunda **raw lag** (F1=0.640994, F2=0.647039).
4. Flattened LightGBM canonical modeli geçti mi? Aynı satırlardaki recorded canonical E0'a göre **True** (E0=0.772760, F1_L12=0.640994). Bu yalnız pilot kanıtıdır.
5. Masked RMSE ne kadar değişti? 0.808143 -> 0.659080; gain=+0.149063.
6. Robust validation kapıları geçildi mi? **Hayır.** Full strict/irregular/spatial/space-time validation çalıştırılmadı; tahmin 93.8 dakika ve 16.6 GB peak RAM, durum `not_run_performance_gate`.
7. GRU pilot kazancının ne kadarı yeniden üretildi? Gösterge oranı **172.8%**; farklı eğitim protokolleri nedeniyle doğrudan nedensel karşılaştırma değildir.
8. GRU sequence mimarisinin ayrı katkısı var mı? **Belirlenemedi.** Aynı full foldlarda kontrollü GRU/flattened karşılaştırması gerekir.
9. Blend işe yaradı mı? Pilot OOF'ta **True**; RMSE=0.640591, canonical weight=0.05. Tek fold olduğu için ağırlık stabilitesi kanıtlanmadı.
10. Submission üretildi mi? **Hayır.** Robust acceptance zorunludur.
11. Full GRU validation hâlâ öncelikli mi? **Evet, ancak önce seçilmiş flattened konfigürasyonun ayrı frozen full validation'ı tamamlanmalı.**
12. İnternet kullanmadan sonraki en güçlü deney nedir? **F1_L12 için tek, frozen strict/irregular/spatial/space-time validation koşusu; yeni feature veya parametre araması yok.**

Pilot promising=True; formal acceptance=False. Notebook 33 GRU değerleri yeniden hesaplanmadı ve `recorded result` olarak etiketlendi. Canonical/current-best submission dosyaları değiştirilmedi.


All required outputs verified; no submission created.
